# Chronic Kidney Disease classification

This notebook is a reproducible educational baseline for the UCI Chronic Kidney Disease dataset.
It deliberately does not contain data or precomputed predictions. Set `CKD_DATA_DIR` to a directory
containing `kidney_disease.csv` before running cells from top to bottom.

**Important:** this is not a clinical decision system. Metrics are estimates on this dataset split and
must not be interpreted as medical advice or evidence of clinical performance.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
DATA_DIR = Path(os.environ.get("CKD_DATA_DIR", "data")).expanduser()
DATA_PATH = DATA_DIR / "kidney_disease.csv"
MODEL_PATH = Path(
    os.environ.get("CKD_MODEL_PATH", "artifacts/kidney_model.joblib")
).expanduser()

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Set CKD_DATA_DIR to the directory containing "
        "kidney_disease.csv, for example: export CKD_DATA_DIR=/absolute/path/to/data"
    )

print(f"Using dataset: {DATA_PATH.resolve()}")


## Load and validate the schema

The public file is commonly distributed with abbreviated UCI column names. The mapping below is
kept in code so the optional `data_description.txt` file is not a runtime dependency. Unknown,
duplicate, or missing required columns fail early with an actionable message.

In [ ]:
RAW_TO_CANONICAL = {
    "id": "id",
    "age": "age",
    "bp": "blood pressure",
    "sg": "specific gravity",
    "al": "albumin",
    "su": "sugar",
    "rbc": "red blood cells",
    "pc": "pus cell",
    "pcc": "pus cell clumps",
    "ba": "bacteria",
    "bgr": "blood glucose random",
    "bu": "blood urea",
    "sc": "serum creatinine",
    "sod": "sodium",
    "pot": "potassium",
    "hemo": "haemoglobin",
    "pcv": "packed cell volume",
    "wc": "white blood cell count",
    "rc": "red blood cell count",
    "htn": "hypertension",
    "dm": "diabetes mellitus",
    "cad": "coronary artery disease",
    "appet": "appetite",
    "pe": "pedal edema",
    "ane": "anemia",
    "class": "classification",
}

NUMERIC_COLUMNS = [
    "age",
    "blood pressure",
    "specific gravity",
    "albumin",
    "sugar",
    "blood glucose random",
    "blood urea",
    "serum creatinine",
    "sodium",
    "potassium",
    "haemoglobin",
    "packed cell volume",
    "white blood cell count",
    "red blood cell count",
]
CATEGORICAL_COLUMNS = [
    "red blood cells",
    "pus cell",
    "pus cell clumps",
    "bacteria",
    "hypertension",
    "diabetes mellitus",
    "coronary artery disease",
    "appetite",
    "pedal edema",
    "anemia",
]
TARGET = "classification"
FEATURE_COLUMNS = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
REQUIRED_COLUMNS = set(FEATURE_COLUMNS + [TARGET])


def normalise_column_name(name):
    return str(name).strip().lower().replace(" ", "_")


def canonicalise_column_name(name):
    cleaned = str(name).strip()
    return RAW_TO_CANONICAL.get(normalise_column_name(cleaned), cleaned)


def load_and_clean_dataset(path):
    frame = pd.read_csv(path)
    frame.columns = [canonicalise_column_name(column) for column in frame.columns]
    duplicate_columns = sorted(frame.columns[frame.columns.duplicated()].unique().tolist())
    if duplicate_columns:
        raise ValueError(f"Dataset contains duplicate columns after alias mapping: {duplicate_columns}")
    missing = sorted(REQUIRED_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError("Dataset is missing required columns: " + ", ".join(missing))

    frame = frame.drop(columns=["id"], errors="ignore").copy()
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    for column in CATEGORICAL_COLUMNS:
        frame[column] = (
            frame[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace({"?": pd.NA, "": pd.NA})
        )

    target_values = (
        frame[TARGET]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({"not ckd": "notckd"})
    )
    target = target_values.map({"ckd": 1, "notckd": 0})
    unknown = sorted(
        target_values[target_values.notna() & ~target_values.isin(["ckd", "notckd"])].dropna().unique().tolist()
    )
    # Keep invalid/missing labels from silently disappearing: dropping target rows
    # would change the declared data contract and can hide a corrupt export.
    if unknown or target.isna().any():
        invalid = sorted(target_values[target.isna()].fillna("<missing>").unique().tolist())
        raise ValueError(
            f"Unsupported or missing target labels: {invalid}. Expected ckd and notckd."
        )
    if target.nunique() != 2:
        raise ValueError("The target must contain both CKD classes after cleaning.")
    return frame.drop(columns=[TARGET]), target.astype("int64")


X, y = load_and_clean_dataset(DATA_PATH)
print(f"Rows: {len(X):,}; features: {X.shape[1]}; class counts: {y.value_counts().sort_index().to_dict()}")

## Split before fitting any preprocessing

The hold-out test set is stratified and never used to fit imputers, encoders, feature selection,
or the classifier. All transformations live inside the scikit-learn pipeline, preventing leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=SEED,
    stratify=y,
)

MIN_CV_SPLITS = 5
minority_train_count = int(y_train.value_counts().min())
if minority_train_count < MIN_CV_SPLITS:
    raise ValueError(
        f"At least {MIN_CV_SPLITS} training rows are required for each class; "
        f"found {minority_train_count}. Provide more labelled data or adjust the split."
    )

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([("imputer", SimpleImputer(strategy="median"))]),
            NUMERIC_COLUMNS,
        ),
        (
            "categorical",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            CATEGORICAL_COLUMNS,
        ),
    ],
    remainder="drop",
)
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "xgboost is required for this notebook. Install dependencies with `pip install -r requirements.txt`."
    ) from exc

classifier = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=1,
    verbosity=0,
)
model = Pipeline([("preprocess", preprocessor), ("classifier", classifier)])

## Fit, cross-validate on training data, and evaluate once on hold-out data

Cross-validation is used only within the training partition. The test metrics are reported once
after model selection; no accuracy claim is hard-coded in this repository.

In [ ]:
cv = StratifiedKFold(n_splits=MIN_CV_SPLITS, shuffle=True, random_state=SEED)
scoring = {
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}
cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise",
)
cv_summary = pd.DataFrame(
    {metric: [cv_results[f"test_{metric}"].mean(), cv_results[f"test_{metric}"].std()] for metric in scoring},
    index=["mean", "std"],
).T
print(cv_summary.to_string())

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_score),
}
pd.Series(test_metrics, name="holdout_test")


In [ ]:
print(classification_report(y_test, y_pred, target_names=["notckd", "ckd"], zero_division=0))


## Optional diagnostic plot

This plot is descriptive only; it does not replace calibration, external validation, or clinical review.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set(
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=["notckd", "ckd"],
    yticklabels=["notckd", "ckd"],
    xlabel="Predicted",
    ylabel="Actual",
    title="Hold-out confusion matrix",
)
for (row, column), value in np.ndenumerate(cm):
    ax.text(column, row, int(value), ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()


## Save the fitted pipeline (optional)

The saved object includes preprocessing and the estimator, so inference uses the same transformations as training.
Only load artifacts from a trusted source because `joblib.load` can execute arbitrary Python code.

In [ ]:
import joblib

ARTIFACT_VERSION = 1
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "artifact_version": ARTIFACT_VERSION,
        "model": model,
        "seed": SEED,
        "numeric_columns": NUMERIC_COLUMNS,
        "categorical_columns": CATEGORICAL_COLUMNS,
        "target_mapping": {0: "notckd", 1: "ckd"},
    },
    MODEL_PATH,
)
print(f"Saved model pipeline to {MODEL_PATH.resolve()}")

## Inference helper

Pass a mapping with the canonical feature names listed in `FEATURE_COLUMNS`. The helper intentionally
does not include a fabricated patient example; callers must supply a real, appropriately governed input record.

In [ ]:
from collections.abc import Mapping


def load_model_artifact(path=MODEL_PATH):
    """Load and validate a trusted fitted-model artifact."""
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Model artifact not found at {path}")
    bundle = joblib.load(path)
    expected = {"artifact_version", "model", "numeric_columns", "categorical_columns", "target_mapping"}
    missing = sorted(expected - set(bundle)) if isinstance(bundle, dict) else sorted(expected)
    if missing:
        raise ValueError(f"Model artifact is missing required keys: {missing}")
    if bundle["artifact_version"] != ARTIFACT_VERSION:
        raise ValueError(
            f"Unsupported model artifact version {bundle['artifact_version']}; expected {ARTIFACT_VERSION}."
        )
    if bundle["numeric_columns"] != NUMERIC_COLUMNS or bundle["categorical_columns"] != CATEGORICAL_COLUMNS:
        raise ValueError("Model artifact feature schema does not match this notebook.")
    if not hasattr(bundle["model"], "predict_proba"):
        raise ValueError("Model artifact does not contain a probabilistic estimator.")
    return bundle


def predict_patient(features, fitted_model=model):
    """Predict one canonical feature mapping; returns 0 (notckd) or 1 (ckd)."""
    if not isinstance(features, Mapping):
        raise TypeError("features must be a mapping of canonical feature names to values")
    row = pd.DataFrame([dict(features)])
    missing = sorted(set(FEATURE_COLUMNS) - set(row.columns))
    extra = sorted(set(row.columns) - set(FEATURE_COLUMNS))
    if missing:
        raise ValueError(f"Missing feature columns: {missing}")
    if extra:
        raise ValueError(f"Unexpected feature columns: {extra}")
    probability = float(fitted_model.predict_proba(row[FEATURE_COLUMNS])[0, 1])
    label = int(probability >= 0.5)
    return {"label": label, "label_name": {0: "notckd", 1: "ckd"}[label], "probability": probability}